In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.romanova.lesson3 import Exercise

print("Загружаем данные...")
digits = load_digits()
X = digits.data.astype(np.float32) / 16.0
y = digits.target
print(f"Всего: {len(X)} образцов, {X.shape[1]} признаков, {len(np.unique(y))} классов")

# делим на train/val/test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")


# функция для обучения одной модели
def train_and_eval(neurons, lr, epochs, batch, X_tr, y_tr, X_val, y_val):
    rng = np.random.default_rng(42)
    model = Exercise.create_model(
        Exercise.create_linear_layer(64, int(neurons), rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(int(neurons), 10, rng),
        Exercise.create_logsoftmax_layer(),
    )
    loss_fn = Exercise.create_nll_loss()
    Exercise.train_model(model, loss_fn, X_tr, y_tr, float(lr), int(epochs), int(batch))
    log_probs = model.forward(X_val)
    preds = np.argmax(log_probs, axis=1)
    return float(np.mean(preds == y_val))


# сетка параметров
print("\nНачинаем перебор параметров...")

neurons_list = [64, 128, 256]
lr_list = [0.001, 0.005, 0.01, 0.05]
epochs_list = [30, 50, 80]
batch_list = [32, 64, 128]

best_acc = -1.0
best_params = None
results = []

total_combos = len(neurons_list) * len(lr_list) * len(epochs_list) * len(batch_list)
counter = 0

for n in neurons_list:
    for lr in lr_list:
        for ep in epochs_list:
            for bs in batch_list:
                counter += 1
                print(f"[{counter}/{total_combos}] пробуем: нейроны={n}, lr={lr}, эпохи={ep}, батч={bs}", end=" ... ")

                acc = train_and_eval(n, lr, ep, bs, X_train, y_train, X_val, y_val)
                results.append({"neurons": n, "lr": lr, "epochs": ep, "batch": bs, "val_acc": acc})

                print(f"точность={acc:.4f}")

                if acc > best_acc:
                    best_acc = acc
                    best_params = {"n_neurons": int(n), "lr": float(lr), "n_epoch": int(ep), "batch_size": int(bs)}
                    print(f"  >>> НОВЫЙ ЛУЧШИЙ! {best_acc:.4f}")

# проверяем, что нашли хотя бы один параметр
if best_params is None:
    print("ОШИБКА: не найдено ни одной валидной конфигурации!")
    exit(1)

# выводим результаты
print(f"\n{'=' * 50}")
print(f"Лучшая точность на валидации: {best_acc:.4f}")
print(f"Лучшие параметры: {best_params}")

print("\nТоп-5 лучших конфигураций:")
results_sorted = sorted(results, key=lambda x: x["val_acc"], reverse=True)
for i in range(min(5, len(results_sorted))):
    r = results_sorted[i]
    print(f"{i + 1}. {r['val_acc']:.4f} | нейроны={r['neurons']}, lr={r['lr']}, эпохи={r['epochs']}, батч={r['batch']}")

# финальное обучение
print("\nФинальное обучение")
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])

rng_final = np.random.default_rng(42)

# Явно преобразуем параметры к нужным типам
final_neurons = int(best_params["n_neurons"])
final_lr = float(best_params["lr"])
final_epochs = int(best_params["n_epoch"])
final_batch = int(best_params["batch_size"])

final_model = Exercise.create_model(
    Exercise.create_linear_layer(64, final_neurons, rng_final),
    Exercise.create_relu_layer(),
    Exercise.create_linear_layer(final_neurons, 10, rng_final),
    Exercise.create_logsoftmax_layer(),
)
loss_fn = Exercise.create_nll_loss()

Exercise.train_model(
    final_model,
    loss_fn,
    X_train_val,
    y_train_val,
    final_lr,
    final_epochs,
    final_batch,
)

# тестирование
test_log_probs = final_model.forward(X_test)
preds = np.argmax(test_log_probs, axis=1)
test_acc = float(np.mean(preds == y_test))

print(f"\n{'=' * 50}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Val accuracy:  {best_acc:.4f}")
print(f"Gap: {abs(test_acc - best_acc):.4f}")

# матрица ошибок
print("\nМатрица ошибок:")
cm = np.zeros((10, 10), dtype=int)
for i in range(len(y_test)):
    cm[y_test[i], preds[i]] += 1

cm_df = pd.DataFrame(cm, index=[f"True_{j}" for j in range(10)], columns=[f"Pred_{j}" for j in range(10)])
print(cm_df)

# точность по каждому классу
print("\nАнализ по классам:")
for digit in range(10):
    total = int(cm[digit].sum())
    correct = int(cm[digit, digit])
    acc_class = correct / total if total > 0 else 0.0
    print(f"Цифра {digit}: {correct}/{total} правильно ({acc_class * 100:.1f}%)")